# E-commerce Data Cleaning with Python & Pandas
**Portfolio project | Synthetic sample data**

This notebook demonstrates a repeatable cleaning workflow using deliberately messy e-commerce data. The source contains duplicate rows, missing values, inconsistent text formatting, and invalid numeric values.

## Objectives
- Inspect data quality and structure.
- Remove duplicates and standardize text.
- Detect and correct invalid values.
- Handle missing values using documented rules.
- Export a cleaned dataset and quality summary.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("messy_ecommerce_data.csv")
df.head()

## 1. Initial inspection

In [ ]:
print("Shape:", df.shape)
display(df.isna().sum().sort_values(ascending=False).to_frame("Missing values"))
print("Duplicate rows:", df.duplicated().sum())
df.info()

## 2. Remove duplicates and standardize text

In [ ]:
clean = df.drop_duplicates().copy()

for col in ["Customer_Name", "Product", "City"]:
    clean[col] = clean[col].astype("string").str.strip().str.title()

clean["Email"] = clean["Email"].astype("string").str.strip().str.lower()

## 3. Validate numeric values

In [ ]:
clean["Age"] = pd.to_numeric(clean["Age"], errors="coerce")
clean.loc[~clean["Age"].between(18, 100), "Age"] = np.nan

clean["Quantity"] = pd.to_numeric(clean["Quantity"], errors="coerce")
clean.loc[clean["Quantity"] <= 0, "Quantity"] = np.nan

clean["Unit_Price"] = pd.to_numeric(clean["Unit_Price"], errors="coerce")
clean.loc[clean["Unit_Price"] < 0, "Unit_Price"] = np.nan

## 4. Handle missing values
Numeric fields use the median. Missing categorical/contact fields are labeled `Not Provided`.

In [ ]:
for col in ["Age", "Quantity", "Unit_Price"]:
    clean[col] = clean[col].fillna(clean[col].median())

clean["Age"] = clean["Age"].round().astype(int)
clean["Quantity"] = clean["Quantity"].round().astype(int)
clean["Unit_Price"] = clean["Unit_Price"].round(2)

for col in ["Email", "Product", "City"]:
    clean[col] = clean[col].fillna("Not Provided")

## 5. Create a calculated field and verify results

In [ ]:
clean["Order_Date"] = pd.to_datetime(clean["Order_Date"], errors="coerce")
clean["Total_Amount"] = (clean["Quantity"] * clean["Unit_Price"]).round(2)

print("Cleaned shape:", clean.shape)
print("Remaining missing cells:", clean.isna().sum().sum())
print("Remaining duplicates:", clean.duplicated().sum())
display(clean.head())

## 6. Export deliverables

In [ ]:
clean["Order_Date"] = clean["Order_Date"].dt.strftime("%Y-%m-%d")
clean.to_csv("cleaned_ecommerce_data.csv", index=False)

summary = pd.DataFrame({
    "Metric": ["Rows before cleaning", "Duplicate rows removed", "Rows after cleaning",
               "Missing cells before", "Missing cells after"],
    "Result": [len(df), int(df.duplicated().sum()), len(clean),
               int(df.isna().sum().sum()), int(clean.isna().sum().sum())]
})
summary.to_csv("cleaning_summary.csv", index=False)
summary

## Notes & assumptions
- This is a demonstration project using synthetic data, not a real client dataset.
- Median imputation is a baseline; real projects should select rules based on business context.
- `Not Provided` avoids inventing customer contact or category information.
- Confirm field meanings and validation rules with a client before processing real data.